# Kannada-MNIST

## Baseline

Train a small two-block CNN for **10 epochs**, using only the training portion of `train.csv`. No Dig-MNIST images or competition test images are used for fitting, augmentation, or checkpoint selection.

- [Competition](https://www.kaggle.com/competitions/Kannada-MNIST)
- [Data](https://www.kaggle.com/competitions/Kannada-MNIST/data)
- [Rules and code-submission requirements](https://www.kaggle.com/competitions/Kannada-MNIST/rules)

### Fixed evaluation protocol

| Data | Role |
| --- | --- |
| 80% of `train.csv` | Fit model parameters |
| 20% of `train.csv` | Stratified validation with split seed **42** |
| 80% of `Dig-MNIST.csv` | Reserved for the mixed experiment; not used in baseline training |
| 20% of `Dig-MNIST.csv` | Same held-out Dig evaluation subset as mixed |
| `test.csv` | Predict labels only; no ground truth is provided |

The split uses original CSV row positions and labels. The split settings are defined once in the split cell below. The seed, validation fraction, stratification labels, and original CSV row order define this partition; no external configuration file is required. Saved split row positions and a SHA-256 hash of `train.csv` identify the exact evaluated partition. Do not reorder, filter, or concatenate rows before splitting.

Report validation and Dig-MNIST scores separately. Validation is used for monitoring; this initial experiment always evaluates the final epoch, with no early stopping or best-checkpoint selection. This notebook generates a submission CSV but never uploads it. GPU arithmetic can differ slightly across runs even when the split is identical.

Dig-MNIST is split separately with the same seed and stratification. Baseline does not fit on either Dig partition; final Dig scores use only the 2,048-row validation partition for held-out evaluation.


Cross-experiment results and interpretations are collected in [report.ipynb](report.ipynb).

In [ ]:
import os
import sys
import time
import random
from pathlib import Path
import hashlib
import json
from importlib.metadata import version

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

# fix seed to reproduce results

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

# resource check
print("Host:", os.uname().nodename)
print("Python:", sys.version)
print("Interpreter:", sys.executable)

assert torch.cuda.is_available()
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

print(
    {
        p: version(p)
        for p in ["torch", "numpy", "pandas", "scikit-learn", "plotly", "kagglehub"]
    }
)

probe = torch.arange(16, dtype=torch.float32, device=device)
assert (probe @ probe).item() == 1240.0
print("CUDA computation passed")

# Embed Plotly assets so notebook outputs can be viewed without a CDN.
pio.renderers.default = "notebook"
pio.templates.default = "plotly_white"


## Data preparation

Use KaggleHub for missing CSVs. Reuse local files, keep original row order, and preserve sample-submission IDs.

In [ ]:
import kagglehub
from dotenv import load_dotenv

COMPETITION = "Kannada-MNIST"
repo_dir = next(
    (
        parent
        for parent in (Path.cwd(), *Path.cwd().parents)
        if (parent / "competitions" / COMPETITION).is_dir()
    ),
    None,
)
project_dir = (
    repo_dir / "competitions" / COMPETITION
    if repo_dir is not None
    else Path.cwd() if Path.cwd().name == COMPETITION else Path.cwd() / COMPETITION
)
output_dir = project_dir / "output"

if repo_dir is not None:
    load_dotenv(repo_dir / ".env", override=False)

required_files = ("train.csv", "test.csv", "sample_submission.csv", "Dig-MNIST.csv")
data_dir = project_dir / "data"
if not all((data_dir / name).is_file() for name in required_files):
    data_dir = Path(
        kagglehub.competition_download(
            COMPETITION,
            output_dir=str(data_dir.resolve()),
        )
    )

missing = [name for name in required_files if not (data_dir / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing competition files in {data_dir}: {missing}")

print("Competition data:", data_dir.resolve())
print("Output directory:", output_dir.resolve())

output_dir = project_dir / "output" / "baseline"
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
train = pd.read_csv(data_dir / "train.csv", dtype=np.int16)
dig_mnist = pd.read_csv(data_dir / "Dig-MNIST.csv", dtype=np.int16)
submission_template = pd.read_csv(data_dir / "sample_submission.csv")
test = pd.read_csv(data_dir / "test.csv")

pixels = [f"pixel{i}" for i in range(784)]
assert train.columns.tolist() == ["label", *pixels]
assert dig_mnist.columns.tolist() == ["label", *pixels]
assert submission_template.columns.tolist() == ["id", "label"]
assert test.columns.tolist() == ["id", *pixels]
assert len(test) == len(submission_template)
assert test["id"].is_unique
assert test["id"].equals(submission_template["id"])

for name, frame in [("train", train), ("Dig-MNIST", dig_mnist), ("test", test)]:
    assert not frame.isna().any().any(), f"Missing values in {name}"
    assert frame[pixels].min().min() >= 0 and frame[pixels].max().max() <= 255
    if "label" in frame:
        assert frame["label"].between(0, 9).all()

display(
    pd.DataFrame(
        [
            {"file": "train.csv", "rows": len(train), "columns": len(train.columns)},
            {
                "file": "Dig-MNIST.csv",
                "rows": len(dig_mnist),
                "columns": len(dig_mnist.columns),
            },
            {"file": "test.csv", "rows": len(test), "columns": len(test.columns)},
            {
                "file": "sample_submission.csv",
                "rows": len(submission_template),
                "columns": len(submission_template.columns),
            },
        ]
    )
)
display(train.head())
display(submission_template.head())


## Reproducible 80/20 splits

Split TRAIN and Dig-MNIST independently, stratifying by label with seed 42 and preserving original CSV row order. Both notebooks must produce identical partitions. Save row positions and both input-file hashes in the split manifest.

- TRAIN: 48,000 learning rows and 12,000 validation rows.
- Dig-MNIST: 8,192 learning rows and 2,048 validation rows.
- Baseline reserves the Dig learning partition without using it for fitting; mixed adds only that partition.


In [ ]:
# Define the split once here; use the same settings and CSV row order in other notebooks.
split_config = {
    "seed": 42,
    "validation_fraction": 0.2,
    "stratify_column": "label",
    "row_order": "original CSV order",
}
all_indices = np.arange(len(train))
train_indices, valid_indices = train_test_split(
    all_indices,
    test_size=split_config["validation_fraction"],
    random_state=split_config["seed"],
    stratify=train[split_config["stratify_column"]].to_numpy(),
)
assert not np.intersect1d(train_indices, valid_indices).size
assert np.array_equal(
    np.sort(np.concatenate([train_indices, valid_indices])), all_indices
)
assert len(train_indices) == int(len(train) * 0.8)

all_dig_indices = np.arange(len(dig_mnist))
dig_train_indices, dig_valid_indices = train_test_split(
    all_dig_indices,
    test_size=split_config["validation_fraction"],
    random_state=split_config["seed"],
    stratify=dig_mnist[split_config["stratify_column"]].to_numpy(),
)
assert not np.intersect1d(dig_train_indices, dig_valid_indices).size
assert np.array_equal(
    np.sort(np.concatenate([dig_train_indices, dig_valid_indices])), all_dig_indices
)

dig_csv_sha256 = hashlib.sha256((data_dir / "Dig-MNIST.csv").read_bytes()).hexdigest()
train_csv_sha256 = hashlib.sha256((data_dir / "train.csv").read_bytes()).hexdigest()
split_manifest = {
    **split_config,
    "train_csv_sha256": train_csv_sha256,
    "dig_csv_sha256": dig_csv_sha256,
    "dig_training_rows": dig_train_indices.tolist(),
    "dig_validation_rows": dig_valid_indices.tolist(),
    "scikit_learn_version": version("scikit-learn"),
    "training_rows": train_indices.tolist(),
    "validation_rows": valid_indices.tolist(),
}
(output_dir / "split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2) + "\n"
)
split_counts = pd.DataFrame(
    {
        "TRAIN learning": train.iloc[train_indices]["label"]
        .value_counts()
        .sort_index(),
        "TRAIN validation": train.iloc[valid_indices]["label"]
        .value_counts()
        .sort_index(),
        "Dig learning": dig_mnist.iloc[dig_train_indices]["label"]
        .value_counts()
        .sort_index(),
        "Dig validation": dig_mnist.iloc[dig_valid_indices]["label"]
        .value_counts()
        .sort_index(),
    }
)
display(split_counts)
print("Training rows:", len(train_indices), "Validation rows:", len(valid_indices))
print("Split seed:", split_config["seed"])

print(
    "Dig learning rows:",
    len(dig_train_indices),
    "Dig validation rows:",
    len(dig_valid_indices),
)


## CNN training

Two convolution/pooling blocks and a dense classifier, with Adam at learning rate 0.001, batch size 256, and 10 fixed epochs. No image augmentation or Dig-MNIST training in this first baseline.

Create one `TensorDataset` from TRAIN, select the fixed stratified row indices with `Subset`, and pass those subsets to `DataLoader`. Training and validation share the same underlying image tensors; only the training loader shuffles. Evaluation loaders preserve row order.

In [ ]:
# Shared tensors with independent, fixed stratified splits for each data source.
dataset = TensorDataset(
    torch.from_numpy(train[pixels].to_numpy(dtype=np.float32)).reshape(-1, 1, 28, 28)
    / 255.0,
    torch.from_numpy(train["label"].to_numpy(dtype=np.int64)),
)
base_train_dataset = Subset(dataset, train_indices.tolist())
valid_dataset = Subset(dataset, valid_indices.tolist())
y_valid = dataset.tensors[1][valid_indices].numpy()

dig_dataset = TensorDataset(
    torch.from_numpy(dig_mnist[pixels].to_numpy(dtype=np.float32)).reshape(
        -1, 1, 28, 28
    )
    / 255.0,
    torch.from_numpy(dig_mnist["label"].to_numpy(dtype=np.int64)),
)
dig_train_dataset = Subset(dig_dataset, dig_train_indices.tolist())
dig_valid_dataset = Subset(dig_dataset, dig_valid_indices.tolist())

train_dataset = base_train_dataset
print(
    "Learning rows:",
    len(train_dataset),
    "TRAIN validation:",
    len(valid_dataset),
    "Dig validation:",
    len(dig_valid_dataset),
)


def make_loader(dataset, shuffle=False):
    return DataLoader(
        dataset,
        batch_size=256,
        shuffle=shuffle,
        num_workers=0,
        generator=torch.Generator().manual_seed(SEED),
    )


train_loader = make_loader(train_dataset, shuffle=True)
valid_loader = make_loader(valid_dataset)

model = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS = 10
history = []
metric_history = []

for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    model.eval()
    validation_loss = 0.0
    probability_batches = []
    with torch.inference_mode():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            validation_loss += criterion(logits, labels).item() * len(labels)
            probability_batches.append(logits.softmax(dim=1).cpu().numpy())

    valid_probabilities = np.concatenate(probability_batches)
    predictions = valid_probabilities.argmax(axis=1)
    class_precision, class_recall, class_f1, _ = precision_recall_fscore_support(
        y_valid, predictions, labels=np.arange(10), zero_division=0
    )
    precision = float(class_precision.mean())
    recall = float(class_recall.mean())
    f1 = float(class_f1.mean())

    metric_history.append(
        dict(
            epoch=epoch,
            target="Overall (macro)",
            precision=precision,
            recall=recall,
            f1=f1,
        )
    )
    for digit in range(10):
        metric_history.append(
            dict(
                epoch=epoch,
                target=f"Digit {digit}",
                precision=class_precision[digit],
                recall=class_recall[digit],
                f1=class_f1[digit],
            )
        )
    row = dict(
        epoch=epoch,
        loss=total_loss / len(train_dataset),
        validation_loss=validation_loss / len(valid_dataset),
        accuracy=accuracy_score(y_valid, predictions),
        precision=precision,
        recall=recall,
        f1=f1,
        seconds=time.perf_counter() - started,
    )
    history.append(row)
    print(row)

display(pd.DataFrame(history))


## Learning curves

- Compare training and validation loss, then inspect overall validation accuracy.
- Precision, recall, and F1 share a legend across three panels.
  - **Overall (macro):** the unweighted mean across the ten digits; shown initially.
  - **Digit 0–9:** click a legend entry to show or hide that digit in all three panels.
  - Double-click an entry to isolate it; double-click again to restore the other series.
- All scores are computed on the validation split after each epoch. Per-digit precision, recall, and F1 treat that digit as the positive class.


In [ ]:
history_df = pd.DataFrame(history)
assert np.isfinite(history_df.select_dtypes(include="number")).all().all()

loss_data = history_df.melt(
    id_vars="epoch",
    value_vars=["loss", "validation_loss"],
    var_name="Split",
    value_name="Cross-entropy loss",
)
fig = px.line(
    loss_data,
    x="epoch",
    y="Cross-entropy loss",
    color="Split",
    markers=True,
    title="Training and validation loss",
)
fig.update_xaxes(dtick=1, title="Epoch")
fig.show()

fig = px.line(
    history_df,
    x="epoch",
    y="accuracy",
    markers=True,
    title="Overall validation accuracy",
)
fig.update_xaxes(dtick=1, title="Epoch")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.show()

metric_history_df = pd.DataFrame(metric_history)
metric_names = {"precision": "Precision", "recall": "Recall", "f1": "F1"}
targets = ["Overall (macro)"] + [f"Digit {i}" for i in range(10)]
colors = ["#111827"] + px.colors.qualitative.Plotly

fig = make_subplots(
    rows=1, cols=3, subplot_titles=list(metric_names.values()), shared_yaxes=True
)
for target_index, target in enumerate(targets):
    rows = metric_history_df[metric_history_df["target"] == target]
    for column, (metric, label) in enumerate(metric_names.items(), start=1):
        fig.add_trace(
            go.Scatter(
                x=rows["epoch"],
                y=rows[metric],
                mode="lines+markers",
                name=target,
                legendgroup=target,
                showlegend=column == 1,
                visible=True if target_index == 0 else "legendonly",
                line=dict(
                    color=colors[target_index], width=3 if target_index == 0 else 2
                ),
                hovertemplate=f"{target}<br>Epoch %{{x}}<br>{label}: %{{y:.3%}}<extra></extra>",
            ),
            row=1,
            col=column,
        )
fig.update_xaxes(dtick=1, title="Epoch")
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.update_layout(
    title="Validation precision, recall, and F1 by epoch",
    legend=dict(
        title="Click to toggle; double-click to isolate", groupclick="togglegroup"
    ),
    height=450,
)
fig.show()


## Final held-out evaluation

Evaluate the final epoch on the fixed TRAIN validation subset and the fixed 20% Dig-MNIST validation subset. Neither evaluation subset is included in this model's training loader. Use the same rows in both experiments and report both domains separately. These validation scores guide development; repeated model selection on them does not provide an untouched final test.


In [ ]:
def predict_probabilities(dataset):
    probabilities = []
    model.eval()
    with torch.inference_mode():
        for batch in make_loader(dataset):
            images = batch[0].to(device)
            probabilities.append(model(images).softmax(dim=1).cpu().numpy())
    return np.concatenate(probabilities)


def score_predictions(labels, probabilities, dataset):
    predicted = probabilities.argmax(axis=1)
    precision, recall, f1, support = precision_recall_fscore_support(
        labels,
        predicted,
        labels=np.arange(10),
        zero_division=0,
    )
    summary = dict(
        dataset=dataset,
        samples=len(labels),
        accuracy=accuracy_score(labels, predicted),
        precision=float(precision.mean()),
        recall=float(recall.mean()),
        f1=float(f1.mean()),
    )
    per_digit = pd.DataFrame(
        {
            "dataset": dataset,
            "digit": np.arange(10),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    )
    return summary, per_digit


y_dig = dig_dataset.tensors[1][dig_valid_indices].numpy()
dig_probabilities = predict_probabilities(dig_valid_dataset)
evaluation_sets = {
    "Validation": (valid_dataset, y_valid, valid_probabilities),
    "Dig-MNIST validation": (dig_valid_dataset, y_dig, dig_probabilities),
}
summaries, per_digit_frames = [], []
for name, (_, labels, probabilities) in evaluation_sets.items():
    summary, per_digit = score_predictions(labels, probabilities, name)
    summaries.append(summary)
    per_digit_frames.append(per_digit)
    fig = px.imshow(
        confusion_matrix(labels, probabilities.argmax(axis=1), labels=np.arange(10)),
        x=[str(i) for i in range(10)],
        y=[str(i) for i in range(10)],
        text_auto=True,
        color_continuous_scale="Blues",
        labels=dict(x="Predicted digit", y="True digit", color="Samples"),
        title=f"{name}: confusion matrix after epoch {EPOCHS}",
    )
    fig.show()

scores_df = pd.DataFrame(summaries)
per_digit_df = pd.concat(per_digit_frames, ignore_index=True)
display(scores_df)
display(per_digit_df)
comparison_rows = []
for row in summaries:
    comparison_rows.append({**row, "target": "Overall (macro)"})
for row in per_digit_df.to_dict("records"):
    comparison_rows.append({**row, "target": f"Digit {row['digit']}"})
comparison = pd.DataFrame(comparison_rows).melt(
    id_vars=["dataset", "target"],
    value_vars=["precision", "recall", "f1"],
    var_name="Metric",
    value_name="Score",
)
fig = px.bar(
    comparison,
    x="Metric",
    y="Score",
    color="target",
    facet_col="dataset",
    barmode="group",
    category_orders={"target": targets},
    color_discrete_map=dict(zip(targets, colors)),
    title="Held-out TRAIN and Dig-MNIST validation metrics",
)
for trace in fig.data:
    trace.visible = True if trace.name == "Overall (macro)" else "legendonly"
fig.update_yaxes(range=[0, 1], tickformat=".0%")
fig.update_layout(legend=dict(title="Click to toggle; double-click to isolate"))
fig.show()


## Competition test inference and saved artifacts

The unlabeled `test.csv` permits predictions but no local accuracy measurement. Preserve the `id,label` template and save predictions alongside split metadata, training history, evaluation tables, and final weights. No remote upload or competition submission is performed.

On Kaggle, the prediction CSV is written directly to `/kaggle/working/submission.csv` so the competition submission selector can use the plain filename `submission.csv`. Local execution keeps it under `output/baseline/`; other artifacts remain there in both environments.

In [ ]:
test_dataset = TensorDataset(
    torch.from_numpy(test[pixels].to_numpy(dtype=np.float32)).reshape(-1, 1, 28, 28)
    / 255.0,
)
test_probabilities = predict_probabilities(test_dataset)
submission = submission_template.copy()
submission["label"] = test_probabilities.argmax(axis=1)
assert submission.columns.tolist() == ["id", "label"]
assert submission["id"].equals(test["id"])
assert submission["label"].between(0, 9).all()
# Kaggle's competition submitter should receive a root-level output filename.
submission_path = (
    Path("/kaggle/working/submission.csv")
    if repo_dir is None and Path("/kaggle/working").is_dir()
    else output_dir / "submission.csv"
)
submission.to_csv(submission_path, index=False)
print("Competition submission file:", submission_path.resolve())
history_df.to_csv(output_dir / "training_history.csv", index=False)
metric_history_df.to_csv(output_dir / "validation_metrics_by_digit.csv", index=False)
scores_df.to_csv(output_dir / "evaluation_summary.csv", index=False)
per_digit_df.to_csv(output_dir / "evaluation_by_digit.csv", index=False)
torch.save(
    {
        "state_dict": model.state_dict(),
        "epochs": EPOCHS,
        "seed": SEED,
        "split_config": split_config,
        "train_csv_sha256": train_csv_sha256,
        "dig_csv_sha256": dig_csv_sha256,
    },
    output_dir / "model.pt",
)
print("Saved baseline artifacts:", output_dir.resolve())
display(submission.head())


## Most confident validation errors

Show up to ten errors from each held-out validation subset, sorted by predicted-label probability. Captions include numeric labels, Kannada Unicode numerals, and softmax probabilities. Neither set of displayed images was used to fit this model. Softmax probabilities are not calibrated confidence estimates.


In [ ]:
def digit_gallery(images, titles, title, captions_below=False):
    count = len(images)
    columns = 5
    rows = (count + columns - 1) // columns
    fig = make_subplots(
        rows=rows,
        cols=columns,
        subplot_titles=None if captions_below else titles,
        vertical_spacing=0.30 if captions_below else 0.15,
    )
    for index, image in enumerate(images):
        fig.add_trace(
            go.Heatmap(
                z=image.reshape(28, 28),
                colorscale="Gray",
                zmin=0,
                zmax=1,
                showscale=False,
                hovertemplate="x=%{x}, y=%{y}<br>Intensity=%{z:.2f}<extra></extra>",
            ),
            row=index // columns + 1,
            col=index % columns + 1,
        )
    if captions_below:
        for index, caption in enumerate(titles):
            fig.add_annotation(
                text=caption,
                x=0.5,
                y=0,
                xref="x domain",
                yref="y domain",
                xanchor="center",
                yanchor="top",
                yshift=-10,
                showarrow=False,
                align="center",
                font=dict(size=12),
                row=index // columns + 1,
                col=index % columns + 1,
            )

    fig.update_xaxes(visible=False, constrain="domain")
    for index in range(rows * columns):
        axis = "x" if index == 0 else f"x{index + 1}"
        fig.update_yaxes(
            visible=False,
            autorange="reversed",
            scaleanchor=axis,
            scaleratio=1,
            row=index // columns + 1,
            col=index % columns + 1,
        )
    fig.update_layout(
        title=title,
        height=rows * (280 if captions_below else 230) + 100,
        margin=dict(t=80 if captions_below else 110, b=90 if captions_below else 40),
    )
    return fig


kannada_digits = [chr(0x0CE6 + digit) for digit in range(10)]
error_frames = []
for dataset_name, (
    evaluation_dataset,
    labels,
    probabilities,
) in evaluation_sets.items():
    predicted = probabilities.argmax(axis=1)
    confidence = probabilities.max(axis=1)
    wrong = np.flatnonzero(predicted != labels)
    selected = wrong[np.argsort(-confidence[wrong], kind="stable")[:10]]
    if len(selected) == 0:
        print(f"No misclassified samples in {dataset_name}.")
        continue
    errors = pd.DataFrame(
        {
            "dataset": dataset_name,
            "evaluation_position": selected,
            "true_label": labels[selected],
            "predicted_label": predicted[selected],
            "predicted_probability": confidence[selected],
            "true_label_probability": probabilities[selected, labels[selected]],
        }
    )
    error_frames.append(errors)
    display(errors)
    captions = [
        f"True: {labels[i]} ({kannada_digits[labels[i]]}) ({probabilities[i, labels[i]]:.2%})<br>"
        f"Pred: {predicted[i]} ({kannada_digits[predicted[i]]}) ({confidence[i]:.2%})"
        for i in selected
    ]
    digit_gallery(
        np.stack([evaluation_dataset[int(i)][0].numpy() for i in selected]),
        captions,
        f"{dataset_name}: top 10 most confident errors",
        captions_below=True,
    ).show()

if error_frames:
    pd.concat(error_frames, ignore_index=True).to_csv(
        output_dir / "confident_errors.csv", index=False
    )
